In [ ]:
!pip install spacy
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
import numpy as np
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")  # or a custom GloVe model
print("Model loaded!")

class WordEmbeddingExplorer:
    def __init__(self):
        # Create input widgets
        self.word1 = widgets.Text(description="Word 1:", value="king")
        self.word2 = widgets.Text(description="Word 2:", value="queen")
        self.word3 = widgets.Text(description="Word 3:", value="man")
        self.word4 = widgets.Text(description="Word 4:", value="woman")

        # Create operation widgets
        self.operation_input = widgets.Text(
            description="Custom:",
            value="king - man + woman",
            placeholder="e.g., paris - france + italy"
        )

        # Create buttons
        self.analyze_button = widgets.Button(description="Analyze Analogy")
        self.viz_button = widgets.Button(description="Visualize")
        self.vector_button = widgets.Button(description="Show Vectors")
        self.custom_button = widgets.Button(description="Calculate Custom")

        # Bind button clicks
        self.analyze_button.on_click(self.analyze_relationships)
        self.viz_button.on_click(self.visualize_relationships)
        self.vector_button.on_click(self.show_vectors)
        self.custom_button.on_click(self.calculate_custom)

        # Create output area
        self.output = widgets.Output()

        # Create top N slider
        self.top_n = widgets.IntSlider(
            value=10,
            min=5,
            max=50,
            description="Top N:",
            style={"description_width": "initial"}
        )

        # Layout
        self.main_box = widgets.VBox([
            widgets.HBox([self.word1, self.word2]),
            widgets.HBox([self.word3, self.word4]),
            widgets.HBox([self.analyze_button, self.viz_button, self.vector_button]),
            widgets.HBox([self.operation_input, self.custom_button]),
            self.top_n,
            self.output
        ])

        display(self.main_box)

    def get_vector(self, word):
        """Get word vector for a given word using the first token."""
        return nlp(word)[0].vector

    def normalize_vector(self, vec):
        """Normalize a vector."""
        norm = np.linalg.norm(vec)
        if norm == 0:
            return vec
        return vec / norm

    def get_normalized_vector(self, word):
        """Get normalized word vector."""
        return self.normalize_vector(self.get_vector(word))

    def cosine_similarity(self, vec1, vec2):
        """Compute cosine similarity between two vectors."""
        vec1_norm = self.normalize_vector(vec1)
        vec2_norm = self.normalize_vector(vec2)
        return np.dot(vec1_norm, vec2_norm)

    def find_similar_words(self, target_vector, n=10, exclude=None):
        """
        Find n most similar words to target_vector.
        Exclude any words in `exclude` (set of strings) to avoid returning them.
        Filter out stop words to skip high-frequency function words.
        """
        if exclude is None:
            exclude = set()

        similarities = []
        for lexeme in nlp.vocab:
            # Skip if no vector, not alpha, or too short
            if not lexeme.has_vector or not lexeme.text.isalpha() or len(lexeme.text) <= 2:
                continue

            # Skip stop words (like 'and', 'that', etc.)
            if lexeme.is_stop:
                continue

            # Exclude original words if requested
            if lexeme.text.lower() in exclude:
                continue

            similarity = self.cosine_similarity(lexeme.vector, target_vector)
            similarities.append((lexeme.text, similarity))

        # Sort by similarity descending
        similarities = sorted(similarities, key=lambda x: x[1], reverse=True)
        return similarities[:n]

    def show_vectors(self, _):
        """Display raw vectors for the input words."""
        with self.output:
            clear_output()

            for label, widget in [
                ("Word 1", self.word1),
                ("Word 2", self.word2),
                ("Word 3", self.word3),
                ("Word 4", self.word4)
            ]:
                if widget.value:
                    vector = self.get_vector(widget.value)
                    print(f"\n{label} ({widget.value}) vector:")
                    print(f"Shape: {vector.shape}")
                    print(f"Norm: {np.linalg.norm(vector):.4f}")
                    print(f"First 10 components: {vector[:10]}")
                    print(f"Min: {vector.min():.4f}, Max: {vector.max():.4f}")
                    print(f"Mean: {vector.mean():.4f}, Std: {vector.std():.4f}")

    def calculate_custom(self, _):
        """
        Calculate custom vector arithmetic, e.g. "king - man + woman".
        Then find most similar words to that resulting vector.
        """
        with self.output:
            clear_output()

            try:
                expression = self.operation_input.value
                terms = expression.split()

                positive_terms = []
                negative_terms = []

                current_op = '+'
                used_words = set()  # track words to exclude from final results

                for term in terms:
                    if term == '+' or term == '-':
                        current_op = term
                    else:
                        vec = self.get_vector(term)
                        used_words.add(term.lower())  # store to exclude later

                        if current_op == '+':
                            positive_terms.append(vec)
                        else:
                            negative_terms.append(vec)

                # Combine vectors by summation, not averaging
                if positive_terms:
                    positive_sum = np.sum(positive_terms, axis=0)
                else:
                    positive_sum = np.zeros(nlp.vocab.vectors.shape[1])

                if negative_terms:
                    negative_sum = np.sum(negative_terms, axis=0)
                else:
                    negative_sum = np.zeros(nlp.vocab.vectors.shape[1])

                result = positive_sum - negative_sum

                print(f"Results for: {expression}\n")
                similar_words = self.find_similar_words(
                    result,
                    n=self.top_n.value,
                    exclude=used_words
                )

                print("Most similar words (excluding original terms + stop words):")
                for word, sim in similar_words:
                    print(f"{word}: {sim:.4f}")

            except Exception as e:
                print(f"Error: {str(e)}")

    def analyze_relationships(self, _):
        """
        Analyze word relationships in the analogy form:
        word1 : word2 :: word3 : ?
        """
        with self.output:
            clear_output()

            try:
                # Get word vectors
                w1 = self.get_vector(self.word1.value)
                w2 = self.get_vector(self.word2.value)
                w3 = self.get_vector(self.word3.value)

                # 3COSADD approach: target = (w2 - w1 + w3)
                target = w2 - w1 + w3

                # Exclude the original words from the results
                exclude_words = {
                    self.word1.value.lower(),
                    self.word2.value.lower(),
                    self.word3.value.lower(),
                    self.word4.value.lower() if self.word4.value else ''
                }

                # Find similar words
                similar_words = self.find_similar_words(
                    target,
                    n=self.top_n.value,
                    exclude=exclude_words
                )

                print(f"Analogy: {self.word1.value} : {self.word2.value} :: {self.word3.value} : ?")
                print(f"\nTop {self.top_n.value} most similar words (excluding original words + stop words):")
                for word, sim in similar_words:
                    print(f"{word}: {sim:.4f}")

                if self.word4.value:
                    w4 = self.get_vector(self.word4.value)
                    similarity = self.cosine_similarity(w4, target)
                    print(f"\nSimilarity with expected word '{self.word4.value}': {similarity:.4f}")

                # Show relationship vector stats
                print("\nRelationship vector (w2 - w1 + w3) stats:")
                print(f"Norm: {np.linalg.norm(target):.4f}")
                print(f"Mean: {target.mean():.4f}")
                print(f"Std: {target.std():.4f}")

            except Exception as e:
                print(f"Error: {str(e)}")

    def visualize_relationships(self, _):
        """Visualize word relationships in 2D using PCA."""
        with self.output:
            clear_output()

            try:
                # Collect words
                words = [self.word1.value, self.word2.value, self.word3.value]
                if self.word4.value:
                    words.append(self.word4.value)

                vectors = [self.get_vector(w) for w in words]

                # Gather some similar words for each
                context_words = []
                context_vectors = []
                for w in words:
                    similarities = []
                    w_vec = self.get_vector(w)
                    for lex in nlp.vocab:
                        if lex.has_vector and lex.text.isalpha() and len(lex.text) > 2:
                            # Also skip stop words from context
                            if lex.is_stop:
                                continue
                            sim = self.cosine_similarity(w_vec, lex.vector)
                            similarities.append((lex.text, sim))
                    top_similar = sorted(similarities, key=lambda x: x[1], reverse=True)[:5]

                    for cw, _ in top_similar:
                        if cw not in words and cw not in context_words:
                            context_words.append(cw)
                            context_vectors.append(self.get_vector(cw))

                # Combine
                all_words = words + context_words
                all_vectors = vectors + context_vectors

                # PCA to 2D
                pca = PCA(n_components=2)
                vectors_2d = pca.fit_transform(all_vectors)

                plt.figure(figsize=(12, 8))

                # Plot context words
                plt.scatter(vectors_2d[len(words):, 0],
                            vectors_2d[len(words):, 1],
                            c='gray', alpha=0.5, label='Context')

                # Plot main words
                plt.scatter(vectors_2d[:len(words), 0],
                            vectors_2d[:len(words), 1],
                            c='red', label='Input words')

                # Annotations
                for i, word in enumerate(all_words):
                    plt.annotate(word,
                                 (vectors_2d[i, 0], vectors_2d[i, 1]),
                                 xytext=(5, 5),
                                 textcoords='offset points')

                # Optionally draw arrows for relationships
                # (Word1 -> Word2) in blue, (Word3 -> Word4) in green
                if len(words) >= 2:
                    plt.arrow(
                        vectors_2d[0, 0], vectors_2d[0, 1],
                        vectors_2d[1, 0] - vectors_2d[0, 0],
                        vectors_2d[1, 1] - vectors_2d[0, 1],
                        head_width=0.1, head_length=0.1,
                        fc='blue', ec='blue', alpha=0.5, label='Relationship 1'
                    )
                if len(words) >= 4:
                    plt.arrow(
                        vectors_2d[2, 0], vectors_2d[2, 1],
                        vectors_2d[3, 0] - vectors_2d[2, 0],
                        vectors_2d[3, 1] - vectors_2d[2, 1],
                        head_width=0.1, head_length=0.1,
                        fc='green', ec='green', alpha=0.5, label='Relationship 2'
                    )

                plt.title("Word Relationships Visualization (PCA)")
                plt.legend()
                plt.grid(True)
                plt.show()

            except Exception as e:
                print(f"Error: {str(e)}")

explorer = WordEmbeddingExplorer()


Loading spaCy model...
Model loaded!
